<a href="https://colab.research.google.com/github/Zheke32174/scandroid/blob/main/scandroid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# scandroid Colab GPU bridge

Runs Ollama on Colab's free GPU and publishes the tunnel URL to a private GitHub Gist so any agent VM (Claude Code remote, local CLI, Codespace) can offload CPU/RAM/GPU-heavy work to a stronger model. **You** open the notebook; the agent only reads the Gist and connects to the tunnel. ToS-clean by construction (see `AI-PARTICIPANTS-TOS-RULE.md` in the cluster).

**One-time setup (first run only):**
1. Runtime → Change runtime type → **T4 GPU**.
2. Left sidebar → 🔑 Secrets → add:
   - `GITHUB_TOKEN` — GitHub token with `gist` scope (private gist publish/update).
   - `NGROK_TOKEN` — optional, from ngrok.com (more reliable than the cloudflare-quick fallback).
   - `GIST_ID` — leave blank on first run; the cell auto-creates one and prints the ID. Paste it back here on subsequent runs.
3. Run all cells.

**Subsequent runs:** just Run All. Gist updates in place; agents reading the same `GIST_ID` pick up the new tunnel URL automatically.

**Agent side:** `from scandroid import discover, generate; print(generate('hello', gist_id='<id>'))`. The cluster's `colab-watcher` daemon also speaks this protocol and updates `colab_endpoint.txt` for it.

In [ ]:
# Papermill-overridable parameters. Notebook works with all defaults.
MODEL = None        # None = auto-pick by VRAM. Override with e.g. 'qwen2.5:14b'.
GIST_DESCRIPTION = 'scandroid-colab-endpoint'
GIST_PUBLIC = False
KEEPALIVE = True    # Set False for headless runs (e.g. papermill smoke tests).


In [ ]:
# 1. GPU check + auto model selection by available VRAM.
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No GPU. Runtime \u2192 Change runtime type \u2192 T4 GPU.')

gpu_info = gpu.stdout.strip()
print('GPU:', gpu_info)
vram_mb = int(gpu_info.split(',')[1].strip().split()[0])

if MODEL is None:
    if   vram_mb >= 38000: MODEL = 'qwen2.5:32b'
    elif vram_mb >= 22000: MODEL = 'qwen2.5:14b'
    elif vram_mb >= 14000: MODEL = 'qwen2.5:7b'
    else:                  MODEL = 'qwen2.5-coder:3b'

print(f'Using model: {MODEL}')


In [ ]:
# 2. Install Ollama (idempotent; the install script is a no-op if already present).
import subprocess
print('Installing Ollama\u2026')
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('Ollama install failed:\n' + r.stderr[-500:])
print('Ollama installed.')


In [ ]:
# 3. Start Ollama daemon listening on all interfaces (the tunnel proxies into 11434).
import subprocess, time, urllib.request, os

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('Waiting for Ollama\u2026', end='')
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print(' ready.')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    raise RuntimeError('Ollama failed to start')


In [ ]:
# 4. Pull the chosen model (cached across reruns within a runtime).
import subprocess
print(f'Pulling {MODEL}\u2026')
subprocess.run(['ollama', 'pull', MODEL], check=True)
print(f'{MODEL} ready.')


In [ ]:
# 5. Tunnel: ngrok (preferred when NGROK_TOKEN set) -> cloudflare quick tunnel (fallback).
import subprocess, threading, time, re, os

try:
    from google.colab import userdata
    _secret = userdata.get
except Exception:
    _secret = lambda k: os.environ.get(k, '')

NGROK_TOKEN = _secret('NGROK_TOKEN') or ''
tunnel_url  = [None]
tunnel_name = [None]


def _try_ngrok():
    if not NGROK_TOKEN:
        return False
    print('  Trying ngrok\u2026', end='', flush=True)
    subprocess.run('pip install -q pyngrok', shell=True)
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    t = ngrok.connect(11434, 'http')
    tunnel_url[0]  = t.public_url.replace('http://', 'https://')
    tunnel_name[0] = 'ngrok'
    print(f' ok ({tunnel_url[0]})')
    return True


def _try_cloudflare():
    print('  Trying cloudflare quick tunnel\u2026', end='', flush=True)
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
        '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
        shell=True)
    p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def _read():
        for line in p.stdout:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if m and not tunnel_url[0]:
                tunnel_url[0]  = m.group(0)
                tunnel_name[0] = 'cloudflare'
    threading.Thread(target=_read, daemon=True).start()
    for _ in range(60):
        if tunnel_url[0]:
            print(f' ok ({tunnel_url[0]})')
            return True
        time.sleep(1)
    print(' failed.')
    return False


print('Starting tunnel (ngrok \u2192 cloudflare)\u2026')
if not _try_ngrok():
    if not _try_cloudflare():
        raise RuntimeError('All tunnel methods failed')

print('=' * 60)
print(f'  ENDPOINT : {tunnel_url[0]}')
print(f'  TUNNEL   : {tunnel_name[0]}')
print(f'  MODEL    : {MODEL}')
print('=' * 60)


In [ ]:
# 6. Publish (or update) the GitHub Gist that agents read for endpoint discovery.
#    Schema is `colab_endpoint.json` with {url, model, ts, tunnel}; this matches
#    the cluster's colab-watcher daemon and scandroid.bridge.discover().
import urllib.request, json, time

GITHUB_TOKEN = _secret('GITHUB_TOKEN') or ''
GIST_ID      = _secret('GIST_ID')      or ''

if not GITHUB_TOKEN:
    print('No GITHUB_TOKEN \u2014 skipping auto-discovery. Add it to Colab Secrets (gist scope).')
    print(f'Manual:  export SCANDROID_ENDPOINT={tunnel_url[0]}  SCANDROID_MODEL={MODEL}')
else:
    body = json.dumps({
        'url':    tunnel_url[0],
        'model':  MODEL,
        'ts':     time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'tunnel': tunnel_name[0],
    }, indent=2)
    payload = {'files': {'colab_endpoint.json': {'content': body}}}
    headers = {'Authorization': f'token {GITHUB_TOKEN}',
               'Content-Type': 'application/json'}

    if GIST_ID:
        req = urllib.request.Request(
            f'https://api.github.com/gists/{GIST_ID}',
            data=json.dumps(payload).encode(),
            headers={**headers, 'X-HTTP-Method-Override': 'PATCH'},
            method='POST')
    else:
        payload['description'] = GIST_DESCRIPTION
        payload['public']      = GIST_PUBLIC
        req = urllib.request.Request(
            'https://api.github.com/gists',
            data=json.dumps(payload).encode(),
            headers=headers, method='POST')

    with urllib.request.urlopen(req) as r:
        resp = json.loads(r.read())
    gist_id = resp.get('id', GIST_ID)

    print(f'Endpoint published to Gist: {gist_id}')
    if not GIST_ID:
        print()
        print('*** Save this GIST_ID for next runs (Colab Secrets) and for your agents: ***')
        print(f'    {gist_id}')
        print()
        print('Agent usage:')
        print(f'    export SCANDROID_GIST_ID={gist_id}')
        print('    python -c "from scandroid import generate; print(generate(\'hello\'))"')
    GIST_ID = gist_id


In [ ]:
# 7. Self-test: hit the public tunnel URL the way an agent would.
import urllib.request, json

payload = json.dumps({'model': MODEL, 'prompt': 'Reply with: COLAB OK', 'stream': False}).encode()
req = urllib.request.Request(f'{tunnel_url[0]}/api/generate', data=payload,
                              headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(req, timeout=120) as r:
    resp = json.loads(r.read())
print('Test response:', resp.get('response', '').strip())
print('Bridge node ready.')


In [ ]:
# 8. Keep-alive + re-publish on tunnel URL change. Runs forever; stop the cell
#    to release the runtime. Set KEEPALIVE = False (parameters cell) for headless
#    smoke tests where you don't want this to block.
import time, urllib.request, datetime, json

if not KEEPALIVE:
    print('KEEPALIVE disabled \u2014 exiting cell. Tunnel will close when runtime stops.')
else:
    print('Keep-alive active.')
    print(f'  endpoint : {tunnel_url[0]}')
    print(f'  model    : {MODEL}')

    last_published_url = tunnel_url[0]
    while True:
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5)
            tag = datetime.datetime.now().strftime('%H:%M')
            print(f'[{tag}] alive  {tunnel_url[0]}', flush=True)
            if GITHUB_TOKEN and GIST_ID and tunnel_url[0] != last_published_url:
                payload = {'files': {'colab_endpoint.json': {'content': json.dumps({
                    'url':    tunnel_url[0], 'model': MODEL,
                    'ts':     time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                    'tunnel': tunnel_name[0],
                }, indent=2)}}}
                req = urllib.request.Request(
                    f'https://api.github.com/gists/{GIST_ID}',
                    data=json.dumps(payload).encode(),
                    headers={'Authorization': f'token {GITHUB_TOKEN}',
                             'Content-Type': 'application/json',
                             'X-HTTP-Method-Override': 'PATCH'}, method='POST')
                urllib.request.urlopen(req)
                last_published_url = tunnel_url[0]
                print(f'[{tag}] gist updated (url changed)')
        except Exception as e:
            print(f'[{datetime.datetime.now().strftime("%H:%M")}] WARN: {e}')
        time.sleep(300)
